In [8]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
df = pd.read_csv('/content/drive/MyDrive/Dataset/Train123.csv')

In [10]:
df.columns

Index(['id', 'MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'AgriculturalPractices', 'Encroachments',
       'IneffectiveDisasterPreparedness', 'DrainageSystems',
       'CoastalVulnerability', 'Landslides', 'Watersheds',
       'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss',
       'InadequatePlanning', 'PoliticalFactors', 'FloodProbability'],
      dtype='object')

In [11]:
X = df.drop(['id', 'FloodProbability'], axis=1)
y = df['FloodProbability']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (894365, 20)
X_test shape: (223592, 20)


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled X_train shape:", X_train_scaled.shape)
print("Scaled X_test shape:", X_test_scaled.shape)

Scaled X_train shape: (894365, 20)
Scaled X_test shape: (223592, 20)


#Decesion Tree

In [13]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_scaled, y_train)

y_pred_prob = dt.predict(X_test_scaled)

threshold = 0.5
y_pred_class = (y_pred_prob >= threshold).astype(int)
y_true_class = (y_test >= threshold).astype(int)

# Calculate metrics
acc = accuracy_score(y_true_class, y_pred_class)
prec = precision_score(y_true_class, y_pred_class)
rec = recall_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class)
cm = confusion_matrix(y_true_class, y_pred_class)
flood_pct = (y_pred_class.sum() / len(y_pred_class)) * 100

# Print formatted output
print("="*50)
print("DECISION TREE CLASSIFICATION REPORT")
print("="*50)
print(f"{'Accuracy':<20} : {acc:.4f}")
print(f"{'Precision':<20} : {prec:.4f}")
print(f"{'Recall':<20} : {rec:.4f}")
print(f"{'F1-Score':<20} : {f1:.4f}")
print("-"*50)
print("Confusion Matrix:")
print("                 Predicted")
print("                 No Flood  Flood")
print(f"Actual No Flood   {cm[0,0]:<9} {cm[0,1]}")
print(f"Actual Flood      {cm[1,0]:<9} {cm[1,1]}")
print("-"*50)
print(f"Flood predicted percentage in test set : {flood_pct:.2f}%")
print("="*50)

DECISION TREE CLASSIFICATION REPORT
Accuracy             : 0.6565
Precision            : 0.6878
Recall               : 0.6802
F1-Score             : 0.6840
--------------------------------------------------
Confusion Matrix:
                 Predicted
                 No Flood  Flood
Actual No Flood   63653     37738
Actual Flood      39075     83126
--------------------------------------------------
Flood predicted percentage in test set : 54.06%


#Xgboost

In [14]:
!pip install xgboost -q

from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train_scaled, y_train)

y_pred_prob = xgb.predict(X_test_scaled)

y_pred_class = (y_pred_prob >= 0.5).astype(int)
y_true_class = (y_test >= 0.5).astype(int)

acc = accuracy_score(y_true_class, y_pred_class)
prec = precision_score(y_true_class, y_pred_class)
rec = recall_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class)
cm = confusion_matrix(y_true_class, y_pred_class)
flood_pct = (y_pred_class.sum() / len(y_pred_class)) * 100

print("="*50)
print("XGBOOST CLASSIFICATION REPORT")
print("="*50)
print(f"{'Accuracy':<20} : {acc:.4f}")
print(f"{'Precision':<20} : {prec:.4f}")
print(f"{'Recall':<20} : {rec:.4f}")
print(f"{'F1-Score':<20} : {f1:.4f}")
print("-"*50)
print("Confusion Matrix:")
print("                 Predicted")
print("                 No Flood  Flood")
print(f"Actual No Flood   {cm[0,0]:<9} {cm[0,1]}")
print(f"Actual Flood      {cm[1,0]:<9} {cm[1,1]}")
print("-"*50)
print(f"Flood predicted percentage in test set : {flood_pct:.2f}%")
print("="*50)

XGBOOST CLASSIFICATION REPORT
Accuracy             : 0.8291
Precision            : 0.8628
Recall               : 0.8172
F1-Score             : 0.8394
--------------------------------------------------
Confusion Matrix:
                 Predicted
                 No Flood  Flood
Actual No Flood   85514     15877
Actual Flood      22336     99865
--------------------------------------------------
Flood predicted percentage in test set : 51.76%


#Lightgbm

In [15]:
!pip install lightgbm -q

from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
lgbm.fit(X_train_scaled, y_train)

y_pred_prob = lgbm.predict(X_test_scaled)

y_pred_class = (y_pred_prob >= 0.5).astype(int)
y_true_class = (y_test >= 0.5).astype(int)

acc = accuracy_score(y_true_class, y_pred_class)
prec = precision_score(y_true_class, y_pred_class)
rec = recall_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class)
cm = confusion_matrix(y_true_class, y_pred_class)
flood_pct = (y_pred_class.sum() / len(y_pred_class)) * 100

print("="*50)
print("LIGHTGBM CLASSIFICATION REPORT")
print("="*50)
print(f"{'Accuracy':<20} : {acc:.4f}")
print(f"{'Precision':<20} : {prec:.4f}")
print(f"{'Recall':<20} : {rec:.4f}")
print(f"{'F1-Score':<20} : {f1:.4f}")
print("-"*50)
print("Confusion Matrix:")
print("                 Predicted")
print("                 No Flood  Flood")
print(f"Actual No Flood   {cm[0,0]:<9} {cm[0,1]}")
print(f"Actual Flood      {cm[1,0]:<9} {cm[1,1]}")
print("-"*50)
print(f"Flood predicted percentage in test set : {flood_pct:.2f}%")
print("="*50)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.708655 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 367
[LightGBM] [Info] Number of data points in the train set: 894365, number of used features: 20
[LightGBM] [Info] Start training from score 0.504480


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LIGHTGBM CLASSIFICATION REPORT
Accuracy             : 0.8321
Precision            : 0.8660
Recall               : 0.8196
F1-Score             : 0.8422
--------------------------------------------------
Confusion Matrix:
                 Predicted
                 No Flood  Flood
Actual No Flood   85893     15498
Actual Flood      22047     100154
--------------------------------------------------
Flood predicted percentage in test set : 51.72%


In [16]:
!pip install catboost -q

from catboost import CatBoostRegressor

cat = CatBoostRegressor(iterations=100, learning_rate=0.1, random_seed=42, verbose=0)
cat.fit(X_train_scaled, y_train)

y_pred_prob = cat.predict(X_test_scaled)

y_pred_class = (y_pred_prob >= 0.5).astype(int)
y_true_class = (y_test >= 0.5).astype(int)

acc = accuracy_score(y_true_class, y_pred_class)
prec = precision_score(y_true_class, y_pred_class)
rec = recall_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class)
cm = confusion_matrix(y_true_class, y_pred_class)
flood_pct = (y_pred_class.sum() / len(y_pred_class)) * 100

print("="*50)
print("CATBOOST CLASSIFICATION REPORT")
print("="*50)
print(f"{'Accuracy':<20} : {acc:.4f}")
print(f"{'Precision':<20} : {prec:.4f}")
print(f"{'Recall':<20} : {rec:.4f}")
print(f"{'F1-Score':<20} : {f1:.4f}")
print("-"*50)
print("Confusion Matrix:")
print("                 Predicted")
print("                 No Flood  Flood")
print(f"Actual No Flood   {cm[0,0]:<9} {cm[0,1]}")
print(f"Actual Flood      {cm[1,0]:<9} {cm[1,1]}")
print("-"*50)
print(f"Flood predicted percentage in test set : {flood_pct:.2f}%")
print("="*50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00
CATBOOST CLASSIFICATION REPORT
Accuracy             : 0.8327
Precision            : 0.8659
Recall               : 0.8210
F1-Score             : 0.8428
--------------------------------------------------
Confusion Matrix:
                 Predicted
                 No Flood  Flood
Actual No Flood   85851     15540
Actual Flood      21874     100327
--------------------------------------------------
Flood predicted percentage in test set : 51.82%


In [ ]:
from sklearn.svm import SVR

svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train_scaled, y_train)

y_pred_prob = svr.predict(X_test_scaled)

y_pred_class = (y_pred_prob >= 0.5).astype(int)
y_true_class = (y_test >= 0.5).astype(int)

acc = accuracy_score(y_true_class, y_pred_class)
prec = precision_score(y_true_class, y_pred_class)
rec = recall_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class)
cm = confusion_matrix(y_true_class, y_pred_class)
flood_pct = (y_pred_class.sum() / len(y_pred_class)) * 100

print("="*50)
print("SVR (SUPPORT VECTOR REGRESSOR) CLASSIFICATION REPORT")
print("="*50)
print(f"{'Accuracy':<20} : {acc:.4f}")
print(f"{'Precision':<20} : {prec:.4f}")
print(f"{'Recall':<20} : {rec:.4f}")
print(f"{'F1-Score':<20} : {f1:.4f}")
print("-"*50)
print("Confusion Matrix:")
print("                 Predicted")
print("                 No Flood  Flood")
print(f"Actual No Flood   {cm[0,0]:<9} {cm[0,1]}")
print(f"Actual Flood      {cm[1,0]:<9} {cm[1,1]}")
print("-"*50)
print(f"Flood predicted percentage in test set : {flood_pct:.2f}%")
print("="*50)

In [ ]:
from sklearn.ensemble import VotingRegressor

voting_reg = VotingRegressor(estimators=[
    ('dt', dt),
    ('xgb', xgb),
    ('lgbm', lgbm),
    ('cat', cat),
    ('svr', svr)
])

voting_reg.fit(X_train_scaled, y_train)

y_pred_prob = voting_reg.predict(X_test_scaled)

y_pred_class = (y_pred_prob >= 0.5).astype(int)
y_true_class = (y_test >= 0.5).astype(int)

acc = accuracy_score(y_true_class, y_pred_class)
prec = precision_score(y_true_class, y_pred_class)
rec = recall_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class)
cm = confusion_matrix(y_true_class, y_pred_class)
flood_pct = (y_pred_class.sum() / len(y_pred_class)) * 100

print("="*50)
print("VOTING REGRESSOR (ENSEMBLE) CLASSIFICATION REPORT")
print("="*50)
print(f"{'Accuracy':<20} : {acc:.4f}")
print(f"{'Precision':<20} : {prec:.4f}")
print(f"{'Recall':<20} : {rec:.4f}")
print(f"{'F1-Score':<20} : {f1:.4f}")
print("-"*50)
print("Confusion Matrix:")
print("                 Predicted")
print("                 No Flood  Flood")
print(f"Actual No Flood   {cm[0,0]:<9} {cm[0,1]}")
print(f"Actual Flood      {cm[1,0]:<9} {cm[1,1]}")
print("-"*50)
print(f"Flood predicted percentage in test set : {flood_pct:.2f}%")
print("="*50)